# OmniSearch — Exploratory Data Analysis

**Heterogeneous air-ground robotic swarm for wildfire search & rescue**
UC Berkeley MIDS Capstone · DATASCI 210

This notebook explores the aggregated simulation results in
`simulations/data/aggregate/simulations_aggregate_data.csv` — one row per
simulation run, produced by the `simulations/` data-generation pipeline.

It compares six coordination strategies across five communication-dropout
levels and seventeen random seeds (**510 runs total**) on the project's
mission metrics: survivor recall, time-to-verification, hazard exposure,
and UGV travel cost.

**To re-run the EDA after generating new data:** just restart the kernel and
run all cells. The notebook reads the CSV fresh each time, so adding more runs
and re-aggregating is automatically reflected here.


## 1. Setup and load

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

mpl.rcParams.update({
    "figure.dpi": 110,
    "font.size": 10,
    "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True,
})

FOREST, SIENNA = "#1F4D2E", "#C44826"


In [ ]:
# Path to the aggregated CSV. Adjust if running from a different working dir.
# Default assumes the notebook lives in simulations/ (next to data/).
from pathlib import Path

CANDIDATES = [
    Path("data/aggregate/simulations_aggregate_data.csv"),
    Path("simulations/data/aggregate/simulations_aggregate_data.csv"),
    Path("simulations_aggregate_data.csv"),
]
CSV_PATH = next((p for p in CANDIDATES if p.exists()), CANDIDATES[0])
print("Loading:", CSV_PATH.resolve())

df = pd.read_csv(CSV_PATH)
print("shape:", df.shape)
df.head()


## 2. Strategy order and palette

We fix a consistent strategy ordering (best→worst by mean recall) and a color
per strategy, used throughout the notebook.

In [ ]:
ORDER = ["lawnmower", "nearest_candidate", "ant_colony",
         "highest_confidence", "random_walk", "random_action"]
PAL = dict(zip(ORDER, sns.color_palette("turbo", len(ORDER))))

# The six mission metrics + key derived features
SUMMARY_METRICS = ["survivor_recall", "time_to_verification",
                   "false_positive_trips", "hazard_exposure", "ugv_travel_cost"]


## 3. Data quality and sanity checks

In [ ]:
# 3a. Run matrix completeness: strategy x dropout x seed
print("Strategies:", df['strategy'].nunique(),
      "| dropout levels:", sorted(df['comms_dropout'].unique()),
      "| seeds:", df['seed'].nunique())
print("\nRuns per strategy:")
print(df['strategy'].value_counts().sort_index())

# Are all cells of the design filled?
design = df.groupby(['strategy', 'comms_dropout'])['seed'].nunique().unstack()
print("\nSeeds per (strategy x dropout) cell — should all equal 17:")
display(design)


In [ ]:
# 3b. Which columns are constant (fixed experiment parameters)?
const_cols = [c for c in df.columns if df[c].nunique(dropna=False) == 1]
print("Constant columns (the fixed design):")
for c in const_cols:
    print(f"  {c:<24} = {df[c].iloc[0]}")


In [ ]:
# 3c. Missing values. time_to_verification / steps_to_first_found are NaN
# exactly when a run never verified any survivor — that is meaningful, not dirty.
miss = df.isna().sum()
print("Columns with missing values:")
print(miss[miss > 0])

# Confirm: missing time_to_verification  <=>  zero survivors found
check = df.assign(no_verify=df['time_to_verification'].isna(),
                  zero_found=(df['final_found'] == 0))
print("\nMissing TTV aligns with zero survivors found?",
      (check['no_verify'] == check['zero_found']).all())


In [ ]:
# 3d. comms_dropout sanity: observed mean comms uptime should equal 1 - dropout
uptime = df.groupby('comms_dropout')['mean_comms_uptime'].mean()
print("dropout -> mean comms uptime (expect 1 - dropout):")
print(uptime.round(3))


## 4. Strategy performance overview

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 9))

# (a) Mean survivor recall by strategy
m = df.groupby("strategy")["survivor_recall"].mean().reindex(ORDER)
ax[0,0].barh(m.index, m.values, color=[PAL[s] for s in m.index])
ax[0,0].invert_yaxis()
ax[0,0].set_title("(a) Mean survivor recall by strategy")
ax[0,0].set_xlabel("survivor recall (fraction of 5 survivors found)")
for i, v in enumerate(m.values):
    ax[0,0].text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)

# (b) Recall distribution by strategy
data_box = [df.loc[df.strategy==s, "survivor_recall"].values for s in ORDER]
bp = ax[0,1].boxplot(data_box, vert=False, patch_artist=True, tick_labels=ORDER)
for patch, s in zip(bp["boxes"], ORDER):
    patch.set_facecolor(PAL[s]); patch.set_alpha(0.7)
for med in bp["medians"]:
    med.set_color("black")
ax[0,1].invert_yaxis()
ax[0,1].set_title("(b) Survivor recall distribution")
ax[0,1].set_xlabel("survivor recall")

# (c) Hazard exposure by strategy
h = df.groupby("strategy")["hazard_exposure"].mean().reindex(ORDER)
ax[1,0].barh(h.index, h.values, color=[PAL[s] for s in h.index])
ax[1,0].invert_yaxis()
ax[1,0].set_title("(c) Mean hazard exposure (lower = safer)")
ax[1,0].set_xlabel("agent-steps in hazard cells")
for i, v in enumerate(h.values):
    ax[1,0].text(v + 0.5, i, f"{v:.0f}", va="center", fontsize=9)

# (d) Efficiency frontier: recall vs UGV travel cost
g = df.groupby("strategy").agg(recall=("survivor_recall","mean"),
                               cost=("ugv_travel_cost","mean"))
for s in ORDER:
    ax[1,1].scatter(g.loc[s,"cost"], g.loc[s,"recall"], s=200,
                    color=PAL[s], edgecolor="black", zorder=3)
    ax[1,1].annotate(s, (g.loc[s,"cost"], g.loc[s,"recall"]),
                     xytext=(6,4), textcoords="offset points", fontsize=8)
ax[1,1].set_title("(d) Efficiency frontier: recall vs UGV travel cost")
ax[1,1].set_xlabel("mean UGV travel cost  (lower = cheaper)")
ax[1,1].set_ylabel("mean survivor recall  (higher = better)")

fig.suptitle("Strategy Performance Overview  (510 runs)",
             fontsize=15, fontweight="bold", color=FOREST)
fig.tight_layout(rect=[0,0,1,0.97])
plt.show()


**Reading this panel.** The four planning baselines (lawnmower,
nearest_candidate, ant_colony, highest_confidence) cluster tightly around
0.27–0.32 mean recall, while the two naive controls (random_walk,
random_action) sit near zero — a sanity check that the task is non-trivial and
the planning strategies add real value. Panel (d) is the key decision view:
**lawnmower and ant_colony reach the same recall as the others at roughly half
the UGV travel cost**, and lawnmower also has the lowest hazard exposure.

## 5. The headline question — robustness to comms dropout

OmniSearch's central claim is about graceful degradation as the network
deteriorates. Here we plot recall against dropout for each strategy.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5.2))

for s in ORDER:
    sub = df[df.strategy == s]
    gp = sub.groupby("comms_dropout")["survivor_recall"]
    mean, sem = gp.mean(), gp.sem()
    ax[0].plot(mean.index, mean.values, marker="o", color=PAL[s], label=s, lw=2)
    ax[0].fill_between(mean.index, mean - 1.96*sem, mean + 1.96*sem,
                       color=PAL[s], alpha=0.15)
ax[0].set_title("(a) Survivor recall vs comms dropout")
ax[0].set_xlabel("comms dropout rate")
ax[0].set_ylabel("survivor recall (mean ± 95% CI)")
ax[0].legend(fontsize=8, ncol=2)

u = df.groupby("comms_dropout")["mean_comms_uptime"].mean()
ax[1].plot(u.index, u.values, "o-", color=FOREST, lw=2, label="observed uptime")
ax[1].plot([0,0.8],[1.0,0.2], "--", color="gray", label="ideal (1 − dropout)")
ax[1].set_title("(b) Data check: comms uptime tracks 1 − dropout")
ax[1].set_xlabel("comms dropout rate")
ax[1].set_ylabel("mean comms uptime")
ax[1].legend()

fig.suptitle("Robustness to Communication Dropout",
             fontsize=15, fontweight="bold", color=FOREST)
fig.tight_layout(rect=[0,0,1,0.95])
plt.show()


**Finding — and an important caveat.** In the current data the planning
strategies are essentially **flat across dropout** (recall barely changes from
0% to 80% dropout), and `nearest_candidate` is the only one that dips at high
dropout. This is a notable result: it suggests that in the present simulation,
coordination quality is *not yet communication-limited* — the agents are
mostly acting on local information. That is worth flagging to the team, because
the whole OmniSearch thesis is that comms *should* matter. Two readings: either
(1) the strategies are robustly decentralized already, or (2) the current
scenario doesn't stress shared information enough for dropout to bite. The
drone-relay / gossip mechanics are the natural next variable to probe.

## 6. Recall surface and verification speed

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))

piv = df.pivot_table(index="strategy", columns="comms_dropout",
                     values="survivor_recall", aggfunc="mean").reindex(ORDER)
sns.heatmap(piv, annot=True, fmt=".3f", cmap="YlOrRd", ax=ax[0],
            cbar_kws={"label": "mean survivor recall"}, linewidths=0.5)
ax[0].set_title("(a) Recall heatmap: strategy × dropout")
ax[0].set_xlabel("comms dropout rate"); ax[0].set_ylabel("")

verified = df[df["time_to_verification"].notna()]
labels_ttv = [s for s in ORDER if (verified.strategy == s).any()]
data_ttv = [verified.loc[verified.strategy==s, "time_to_verification"].values
            for s in labels_ttv]
bp = ax[1].boxplot(data_ttv, vert=True, patch_artist=True, tick_labels=labels_ttv)
for patch, s in zip(bp["boxes"], labels_ttv):
    patch.set_facecolor(PAL[s]); patch.set_alpha(0.7)
for med in bp["medians"]:
    med.set_color("black")
ax[1].set_title("(b) Time-to-verification (verified runs only)")
ax[1].set_ylabel("steps to verification")
ax[1].tick_params(axis="x", rotation=30)

fig.suptitle("Recall Surface & Verification Speed",
             fontsize=15, fontweight="bold", color=FOREST)
fig.tight_layout(rect=[0,0,1,0.95])
plt.show()


**Reading this.** The heatmap makes the flat-across-dropout pattern
explicit per cell. For verification speed (panel b), among runs that *did*
verify at least one survivor, the median time-to-verification is similar
(~160–210 steps) across strategies — `ant_colony` and `nearest_candidate`
trend slightly faster. Note `random_action` is absent here: it never verifies
anyone, so it has no time-to-verification to plot.

## 7. Correlations among metrics

In [ ]:
# Numeric metrics that actually vary (drop constants and IDs)
num = df.select_dtypes("number")
varying = [c for c in num.columns if num[c].nunique() > 1
           and c not in ("seed",)]
corr = df[varying].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Correlation matrix of varying metrics", fontweight="bold")
plt.tight_layout()
plt.show()


**Reading this.** Look along the `survivor_recall` and `final_found` rows
for what co-moves with success. `comms_dropout` should show a near-zero
correlation with recall here — quantitatively confirming the flat relationship
seen in Section 5. `mean_comms_uptime` is essentially a perfect negative proxy
for `comms_dropout` (it *is* 1 − dropout), so treat that pair as redundant.

## 8. Summary table

In [ ]:
summary = (df.groupby("strategy")
    .agg(recall_mean=("survivor_recall", "mean"),
         recall_std=("survivor_recall", "std"),
         pct_zero_recall=("survivor_recall", lambda s: (s == 0).mean()*100),
         hazard_mean=("hazard_exposure", "mean"),
         ugv_cost_mean=("ugv_travel_cost", "mean"),
         ttv_median=("time_to_verification", "median"))
    .reindex(ORDER).round(3))
summary


In [ ]:
# Save the summary next to the aggregate CSV for the write-up / slides
out = CSV_PATH.parent / "eda_strategy_summary.csv"
summary.to_csv(out)
print("Wrote", out.resolve())


## 9. Takeaways

1. **The task is non-trivial and the planners work.** Random baselines find
   ~0 survivors; the four planning strategies all reach 0.27–0.32 recall.

2. **lawnmower is the current efficiency leader** — top recall (0.315), lowest
   hazard exposure, and among the lowest UGV travel cost. `ant_colony` matches
   it on cost with slightly lower recall.

3. **Recall is surprisingly flat across comms dropout.** In this dataset,
   degrading the network from 0% to 80% barely moves recall for the planning
   strategies. This is the most important thing to discuss with the team: it
   either demonstrates strong decentralization or signals that the scenario
   doesn't yet stress shared information enough for comms to matter. The
   drone-relay / gossip mechanics are the natural next experiment.

4. **`false_positive_trips` is identically zero** across all 510 runs — either
   the metric isn't being exercised by these strategies or it isn't wired up in
   the simulator yet. Worth confirming before relying on it.

5. **Next steps for the EDA:** add seeds/strategies, re-aggregate, and re-run
   this notebook; introduce scenarios that make communication a binding
   constraint; and once HAPPO results exist, add them as a strategy to compare
   against these baselines on the same axes.


---
# Part II — Deeper effectiveness diagnostics

The sections below go beyond means to ask *why* strategies differ, *how
reliably*, and whether the differences are *statistically real*. All cells read
from the same `df`, so they re-run automatically against new data.

In [ ]:
from scipy.stats import mannwhitneyu

def cliffs_delta(a, b):
    """Nonparametric effect size in [-1, 1]. Positive => a tends to exceed b."""
    import numpy as np
    a, b = np.asarray(a), np.asarray(b)
    gt = sum((x > b).sum() for x in a)
    lt = sum((x < b).sum() for x in a)
    return (gt - lt) / (len(a) * len(b))

N_SURV = int(df['n_survivors'].iloc[0])  # 5


## 10. Search → verification funnel (the key diagnostic)

Every survivor must first be **scouted** by a drone, then **found** (verified)
by a UGV. Splitting recall into these two stages localizes *where* a strategy
fails: a high scouted-but-low-found gap is a handoff/verification problem, not a
search problem.

In [ ]:
import numpy as np
fig, ax = plt.subplots(1, 2, figsize=(13, 5.3))

g = df.groupby("strategy")[["final_scouted", "final_found"]].mean().reindex(ORDER)
y = np.arange(len(ORDER))
ax[0].barh(y, g["final_scouted"], color="#9ecae1", label="scouted (drone saw)")
ax[0].barh(y, g["final_found"], color=[PAL[s] for s in ORDER],
           label="found (UGV verified)")
ax[0].set_yticks(y); ax[0].set_yticklabels(ORDER); ax[0].invert_yaxis()
ax[0].set_title("(a) Search vs verification funnel")
ax[0].set_xlabel(f"mean survivors (out of {N_SURV})")
ax[0].legend(loc="lower right", fontsize=9)
for i, s in enumerate(ORDER):
    ax[0].text(g["final_scouted"].iloc[i]+0.03, i, f"{g['final_scouted'].iloc[i]:.2f}", va="center", fontsize=8)
    ax[0].text(g["final_found"].iloc[i]+0.03, i, f"{g['final_found'].iloc[i]:.2f}", va="center", fontsize=8, fontweight="bold")

conv = (g["final_found"] / g["final_scouted"]).fillna(0)
ax[1].barh(y, conv.values, color=[PAL[s] for s in ORDER])
ax[1].set_yticks(y); ax[1].set_yticklabels(ORDER); ax[1].invert_yaxis()
ax[1].set_title("(b) Verification conversion rate (found ÷ scouted)")
ax[1].set_xlabel("fraction of scouted survivors that were verified")
ax[1].set_xlim(0, 1)
for i, v in enumerate(conv.values):
    ax[1].text(v+0.01, i, f"{v:.0%}", va="center", fontsize=9)

fig.suptitle("Where strategies separate: detection → verification handoff",
             fontsize=15, fontweight="bold", color=FOREST)
fig.tight_layout(rect=[0,0,1,0.95]); plt.show()


**Finding.** All strategies — *including random_walk* — scout ~2.3 of 5
survivors. The separation is entirely in **conversion**: the planners verify
60–67% of what they scout, random_walk converts ~2%, random_action 0%. So the
drones' search is *not* the bottleneck in this setup; the **UGV handoff is the
entire game**. That reframes where modeling effort should go.

## 11. Full outcome distribution (not just the mean)

`final_found` is discrete (0–4 here). Showing the share of runs at each outcome
is more honest than a mean: it distinguishes "reliably finds ~1.5" from "usually
0, occasionally 4".

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
counts = (df.groupby("strategy")["final_found"]
          .value_counts(normalize=True).unstack(fill_value=0).reindex(ORDER))
for k in range(5):
    if k not in counts.columns: counts[k] = 0.0
counts = counts[[0,1,2,3,4]]
greens = sns.color_palette("YlGn", 5)
left = np.zeros(len(ORDER))
for k in range(5):
    ax.barh(ORDER, counts[k].values, left=left, color=greens[k], edgecolor="white", label=f"{k} found")
    for i,(val,l) in enumerate(zip(counts[k].values, left)):
        if val > 0.04:
            ax.text(l+val/2, i, f"{val:.0%}", va="center", ha="center", fontsize=8)
    left += counts[k].values
ax.invert_yaxis(); ax.set_xlim(0,1)
ax.set_xlabel("share of the 85 runs per strategy")
ax.set_title("How many of 5 survivors were found — full outcome distribution", fontsize=14, color=FOREST)
ax.legend(ncol=5, loc="lower center", bbox_to_anchor=(0.5,-0.18), fontsize=9)
ax.grid(False); fig.tight_layout(); plt.show()


**Finding.** The planners never find all 5 — the realistic ceiling here
is 3–4. lawnmower has the most 3+ outcomes (16%) and fewest zeros (8%);
highest_confidence piles up at exactly 1 (49%). random_action finds 0 in 100%
of runs, the clean lower bound.

## 12. Skill vs luck — strategy × seed

Seeds are shared across strategies, so a column = one fixed survivor layout +
fire. Vertical stripes (whole columns dark or light) reveal maps that are
intrinsically hard/easy for *everyone*; rows that stay bright through hard
columns indicate genuine robustness rather than lucky maps.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.6))
piv = (df.pivot_table(index="strategy", columns="seed",
                      values="survivor_recall", aggfunc="mean").reindex(ORDER))
sns.heatmap(piv, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax,
            cbar_kws={"label":"mean recall (avg over 5 dropout levels)"},
            linewidths=0.5, annot_kws={"fontsize":7})
seed_mean = piv.mean(axis=0)
hard, easy = seed_mean.idxmin(), seed_mean.idxmax()
ax.set_title("Recall by strategy × seed — vertical stripes = intrinsically hard maps", fontsize=14, color=FOREST)
ax.set_xlabel(f"seed  (hardest = {hard}, easiest = {easy}; shared layout across strategies)")
ax.set_ylabel(""); fig.tight_layout(); plt.show()


**Finding.** Difficulty is largely map-driven: some seeds are hard for
every planner while others are broadly easier. This is why per-strategy means
carry wide standard deviations — a lot of the variance is *which map you drew*,
not the strategy. It also justifies using shared seeds and a nonparametric
paired view when comparing strategies.

## 13. Multi-objective trade-off profile

No single metric captures "best". This parallel-coordinates plot puts five
objectives on one chart, each oriented so **up = better**. Each line is a
strategy's profile — the shape *is* the trade-off.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
prof = df.groupby("strategy").agg(
    recall=("survivor_recall","mean"),
    speed=("time_to_verification","mean"),
    safety=("hazard_exposure","mean"),
    ugv_eff=("ugv_travel_cost","mean"),
    coverage=("total_drone_path_len","mean"),
).reindex(ORDER)
axes_def = [("recall","Survivor\nrecall",False),("speed","Verification\nspeed",True),
            ("safety","Safety\n(low hazard)",True),("ugv_eff","UGV\nefficiency",True),
            ("coverage","Search\ncoverage",False)]
norm = pd.DataFrame(index=prof.index)
for col,_,inv in axes_def:
    v = prof[col].astype(float); lo,hi = v.min(),v.max()
    z = (v-lo)/(hi-lo) if hi>lo else v*0+0.5
    norm[col] = (1-z) if inv else z
xs = np.arange(len(axes_def))
for s in ORDER:
    ax.plot(xs, [norm.loc[s,c] for c,_,_ in axes_def], marker="o", color=PAL[s], lw=2.2, label=s, alpha=0.9)
ax.set_xticks(xs); ax.set_xticklabels([l for _,l,_ in axes_def])
ax.set_ylim(-0.05,1.05); ax.set_ylabel("normalized score  (1 = best across strategies)")
ax.set_title("Multi-objective trade-off profile — each line is one strategy", fontsize=14, color=FOREST)
ax.legend(loc="center left", bbox_to_anchor=(1.01,0.5), fontsize=9)
for x in xs: ax.axvline(x, color="gray", alpha=0.2, lw=1)
fig.tight_layout(); plt.show()


**Finding.** lawnmower dominates on recall, safety, and UGV efficiency but
sits lowest on raw search coverage — it succeeds by being *orderly*, not by
flying farthest. random_walk/random_action rack up the most drone motion yet
convert none of it. "More searching" is clearly not the lever; coordinated
handoff is.

## 14. Are the differences real? Significance + effect size

With 85 runs per strategy, even trivial gaps can look "significant", so we pair
**Mann-Whitney U** p-values (your slide-5 method) with **Cliff's δ** effect
sizes. Read them together: significant *and* non-trivial δ = a difference that
matters.

In [ ]:
strat = ORDER
P = pd.DataFrame(np.nan, index=strat, columns=strat)
D = pd.DataFrame(np.nan, index=strat, columns=strat)
for a in strat:
    ra = df[df.strategy==a]["survivor_recall"].values
    for b in strat:
        rb = df[df.strategy==b]["survivor_recall"].values
        if a==b:
            D.loc[a,b]=0.0
        else:
            _,p = mannwhitneyu(ra,rb,alternative="two-sided")
            P.loc[a,b]=p; D.loc[a,b]=cliffs_delta(ra,rb)

fig, ax = plt.subplots(1,2,figsize=(14,5.8))
annot = P.copy().astype(object)
for a in strat:
    for b in strat:
        v=P.loc[a,b]
        if pd.isna(v): annot.loc[a,b]=""
        else:
            star="***" if v<.001 else "**" if v<.01 else "*" if v<.05 else "ns"
            annot.loc[a,b]=f"{v:.3f}\n{star}"
sns.heatmap(P.astype(float), annot=annot.values, fmt="", cmap="viridis_r", ax=ax[0],
            cbar_kws={"label":"p-value"}, linewidths=0.5, vmin=0, vmax=0.1, annot_kws={"fontsize":7})
ax[0].set_title("(a) Mann-Whitney U p-value (recall)\n* p<.05  ** p<.01  *** p<.001")
sns.heatmap(D.astype(float), annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax[1],
            cbar_kws={"label":"Cliff's δ (row vs col)"}, linewidths=0.5, vmin=-1, vmax=1, annot_kws={"fontsize":8})
ax[1].set_title("(b) Effect size — Cliff's δ\n|δ|: 0.15 small · 0.33 medium · 0.47 large")
for a in ax:
    a.set_xticklabels(a.get_xticklabels(), rotation=35, ha="right", fontsize=8)
    a.set_yticklabels(a.get_yticklabels(), rotation=0, fontsize=8)
fig.suptitle("Are the differences real? Significance and effect size", fontsize=15, fontweight="bold", color=FOREST)
fig.tight_layout(rect=[0,0,1,0.94]); plt.show()


**Finding — important.** Every planner beats both random baselines with
huge effect sizes (δ ≈ 0.8–0.9, p < 0.001). But **the four planners are
statistically indistinguishable from each other** (all p > 0.09, |δ| ≤ 0.14,
small). So the headline ranking ("lawnmower is best") is *not* statistically
supported on recall alone — lawnmower's real edge is on the *secondary* axes
(hazard, UGV cost) seen in §13. The honest claim for the professors is:
**planning beats random decisively; choosing among planners should be driven by
safety/efficiency, not recall.**

## 15. Updated takeaways (Part II)

1. **The UGV handoff is the bottleneck, not search.** Every strategy scouts the
   same ~2.3 survivors; planners differ only in converting scouts to verified
   finds (60–67% vs ~0% for random). Model the handoff, not the search.
2. **Planning ≫ random, but planners ≈ each other on recall.** δ ≈ 0.8–0.9 vs
   random; p > 0.09 among planners. Differentiate planners on hazard and UGV
   efficiency instead.
3. **lawnmower is the best *all-round* profile** — same recall, lower hazard,
   lower UGV cost — but the recall lead itself is within noise.
4. **Difficulty is map-driven.** Shared-seed heatmap shows whole columns
   hard/easy for everyone; report paired comparisons and keep seeds fixed.
5. **Still open:** comms dropout barely bites (Part I §5), and
   `false_positive_trips` is all-zero — both point to simulator mechanics
   (relay/gossip, FP wiring) as the next thing to exercise before the HAPPO
   comparison.